# Stage 1 — Navigation Curriculum

Four-level curriculum training a single robot to navigate a warehouse grid
while managing battery (charge when needed, reach goal without dying).

| Level | Grid  | Obstacles          | Battery starts |
|-------|-------|--------------------|----------------|
| L1    | 5×5   | None               | Full           |
| L2    | 10×10 | None               | Full           |
| L3    | 10×10 | 10 static          | Random 20–100% |
| L4    | 10×10 | 5 static + 3 dyn.  | Random 20–100% |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib
matplotlib.use('Agg')

from envs.nav_env import WarehouseEnv
from agents.ppo import DQN, PPO, transfer_dqn_to_ppo, select_action
from utils.replay_buffer import ReplayBuffer
from utils.plotting import plot_nav_history, plot_nav_full_curriculum
from training.train_nav import train_dqn, train_ppo, DEVICE, CKPT_DIR

print(f'Device: {DEVICE}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, lvl in zip(axes, [1, 2, 3, 4]):
    env = WarehouseEnv(level=lvl)
    env.reset()
    ax.imshow(env.grid, cmap='RdYlGn', vmin=-1, vmax=3)
    ax.set_title(f'Level {lvl} ({env.size}×{env.size})')
    ax.axis('off')
plt.suptitle('Warehouse Environments', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/nav_env_grid.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
print('Training L1 DQN...')
dqn_l1, r1, s1 = train_dqn(level=1, episodes=8_000)
torch.save(dqn_l1.state_dict(), f'{CKPT_DIR}/dqn_l1.pt')
plot_nav_history(r1, s1, level=1, out_dir=CKPT_DIR)

In [ ]:
print('Training L2 DQN...')
dqn_l2, r2, s2 = train_dqn(level=2, episodes=8_000, pretrained=dqn_l1)
torch.save(dqn_l2.state_dict(), f'{CKPT_DIR}/dqn_l2.pt')
plot_nav_history(r2, s2, level=2, out_dir=CKPT_DIR)

In [ ]:
print('Training L3 PPO...')
ppo_l3 = PPO(state_dim=13, action_dim=6).to(DEVICE)
transfer_dqn_to_ppo(dqn_l2, ppo_l3)
ppo_l3, r3, s3 = train_ppo(level=3, model=ppo_l3, episodes=20_000)
torch.save(ppo_l3.state_dict(), f'{CKPT_DIR}/ppo_l3.pt')
plot_nav_history(r3, s3, level=3, out_dir=CKPT_DIR)

In [ ]:
print('Training L4 PPO...')
ppo_l4 = PPO(state_dim=13, action_dim=6).to(DEVICE)
ppo_l4.load_state_dict(ppo_l3.state_dict())
ppo_l4, r4, s4 = train_ppo(level=4, model=ppo_l4, episodes=16_000)
torch.save(ppo_l4.state_dict(), f'{CKPT_DIR}/ppo_final.pt')
print(f'Saved final nav policy → {CKPT_DIR}/ppo_final.pt')
plot_nav_history(r4, s4, level=4, out_dir=CKPT_DIR)

In [ ]:
import numpy as np

plot_nav_full_curriculum({1: r1, 2: r2, 3: r3, 4: r4}, out_dir=CKPT_DIR)
for lvl, arr in zip([1, 2, 3, 4], [r1, r2, r3, r4]):
    np.save(f'{CKPT_DIR}/nav_l{lvl}_rewards.npy', arr)